# Edge Plant Doctor sur Colab (GPU T4)

Ce carnet rejoue le pipeline du depot sur **toute** la base PlantVillage avec un GPU, en appelant les memes commandes que la version CPU (`python -m edgeplant.*`). Aucune logique n'est dupliquee ici.

Menu : *Execution > Modifier le type d'execution > GPU T4*, puis *Tout executer*. Compter environ 40 minutes. A la fin, `colab_artifacts.zip` se telecharge : il contient les modeles ONNX et les JSON de metriques.

Les chiffres de ce carnet viennent d'un run **Colab GPU sur la base complete** et ne se melangent pas avec ceux du run CPU du depot (sous-ensemble de 150 images par classe).

In [ ]:
!nvidia-smi -L
import torch

assert torch.cuda.is_available(), "Activez un GPU : Execution > Modifier le type d'execution"

## 1. Installer le depot public

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/Adam-Blf/edge-plant-doctor.git
%cd /content/edge-plant-doctor
!pip install -q -e .

## 2. Sorties separees du run CPU

Les checkpoints, le cache et les resultats vivent dans `/content`, hors du depot.

In [ ]:
%env EDGEPLANT_CACHE=/content/cache
%env EDGEPLANT_CHECKPOINTS=/content/checkpoints
%env EDGEPLANT_RESULTS=/content/results

## 3. Base complete, decoupe par feuille

Telecharge `mohanty/PlantVillage` (2 Go) puis met en cache les 54 000 images en 224x224. `--full` retire le plafond par classe.

In [ ]:
!python -m edgeplant.data --full

## 4. Enseignant EfficientNet-B0

Les premiers etages sont geles et mis en cache, les autres sont ajustes (meme code que sur CPU).

In [ ]:
!python -m edgeplant.teacher --epochs 8

## 5. Eleve MobileNetV3-Small : reference puis distillation

In [ ]:
!python -m edgeplant.train --arch student --tag student_baseline --epochs 15
!python -m edgeplant.distill --tag student_distilled --epochs 15

## 6. Export ONNX et quantification int8 (dynamique et statique)

In [ ]:
!python -m edgeplant.export --arch student --tag student_baseline
!python -m edgeplant.export --arch student --tag student_distilled
!python -m edgeplant.quantize --tag student_baseline
!python -m edgeplant.quantize --tag student_distilled

## 7. Precision, F1 et taille de chaque variante sur le jeu de test

In [ ]:
!python -m edgeplant.evaluate

## 8. Recuperer les artefacts

Le zip contient les ONNX et les JSON. Pour les ranger dans le depot : decompresser dans `results/colab/` (JSON) et `models/colab/` (ONNX), puis commit. La quantification par entrainement (QAT) n'est pas faite ici.

In [ ]:
import pathlib
import shutil

out = pathlib.Path("/content/colab_artifacts")
shutil.rmtree(out, ignore_errors=True)
(out / "models").mkdir(parents=True)
(out / "results").mkdir()
for f in pathlib.Path("/content/checkpoints").glob("student_*.onnx"):
    shutil.copy(f, out / "models" / f.name)
for f in pathlib.Path("/content/results").glob("*.json"):
    shutil.copy(f, out / "results" / f.name)
shutil.make_archive("/content/colab_artifacts", "zip", out)
from google.colab import files

files.download("/content/colab_artifacts.zip")